In [0]:
use sql_catalog.sql_schema;
select * from emp;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30
7698,BLAKE,MANAGER,7839,1981-05-01,2850,null,30
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30


In [0]:
select max(sal) as highest_sal_only from emp;

highest_sal_only
5000


In [0]:
select deptno, max(sal) as highest_sal_deptwise
from emp
group by deptno
order by deptno asc;


deptno,highest_sal_deptwise
10,5000
20,3000
30,2850


In [0]:
--  Q1: Highest salary in the company
-- The subquery finds the maximum salary, and the outer query returns everyone who earns it. If two people tie, both are shown.

-- Alternative using a window function:

select ename, sal as highest_sal_withName
from emp 
where sal = (select max(sal) from emp);



ename,highest_sal_withName
KING,5000


In [0]:
SELECT ename, sal
FROM (
  SELECT ename, sal,
         DENSE_RANK() OVER (ORDER BY sal DESC) AS rnk
  FROM emp
) t
WHERE rnk = 1;

ename,sal
KING,5000


-- Q2: Highest salary in each department with ename.

This is a correlated subquery: for each employee, it checks whether their salary equals the max of their own department.

Common mistake: SELECT name, dept, MAX(salary) FROM employees GROUP BY dept; fails (or gives wrong names) because name isn’t in the GROUP BY. That’s a very common interview trap, so mention it if asked.

The error comes from the subquery. 
GROUP BY deptno makes it return one max per department, so it returns 3 rows (5000, 3000, 2850). 
The = operator expects a single value, 
so the database throws something like “single-row subquery returns more than one row” (Oracle) 

In [0]:
select e.ename, e.sal as highest_sal_deptWise, e.deptno
from emp e
where e.sal = (select max(sal) from emp group by deptno);

-- Throws error


empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30
7698,BLAKE,MANAGER,7839,1981-05-01,2850,null,30
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30


In [0]:
SELECT * FROM emp WHERE deptno = 10;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7934,MILLER,CLERK,7782,1982-01-23,1300,null,10


In [0]:
UPDATE emp
SET sal = 3000
WHERE ename = 'CLARK';

select * from emp where deptno = 10;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7782,CLARK,MANAGER,7839,1981-06-09,3000,null,10
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7934,MILLER,CLERK,7782,1982-01-23,1300,null,10


In [0]:
select e.ename, e.sal as highest_sal_deptWise, e.deptno
from emp e
where e.sal IN (SELECT MAX(sal) FROM emp GROUP BY deptno);
--- cant use IN here. as This checks whether the salary matches any department’s max, not the max of its own department ONLY.
-- For example, a dept 10 employee earning 3000 would wrongly pass because 3000 is the max of dept 20.

ename,highest_sal_deptWise,deptno
CLARK,3000,10
BLAKE,2850,30
SCOTT,3000,20
KING,5000,10
FORD,3000,20


In [0]:
UPDATE emp
SET sal = 2450
WHERE ename = 'CLARK';

num_affected_rows
1


In [0]:
select e.ename, e.sal as highest_sal_deptWise, e.deptno
from emp e
where e.sal = (select max(sal) from emp where deptno = e.deptno);


ename,highest_sal_deptWise,deptno
BLAKE,2850,30
SCOTT,3000,20
KING,5000,10
FORD,3000,20


In [0]:
-- using join
SELECT e.ename, e.sal AS highest_sal_deptwise, e.deptno
FROM emp e
JOIN (
  SELECT deptno, MAX(sal) AS max_sal
  FROM emp
  GROUP BY deptno
) m ON e.deptno = m.deptno AND e.sal = m.max_sal

ename,highest_sal_deptwise,deptno
BLAKE,2850,30
SCOTT,3000,20
KING,5000,10
FORD,3000,20


Rule to remember: =, <, > need a subquery that returns exactly one value. If it can return many rows, use IN, ANY, or ALL, or correlate the subquery.

In [0]:
select * from emp
where deptno = 20;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7876,ADAMS,CLERK,7788,1983-01-12,1100,null,20
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20


In [0]:
select max(sal) from emp 
where deptno = 20;

max(sal)
3000


select ename, max(sal) from emp 
where deptno = 20; 


--- ERROR
group by [required](url)

Interview tip: Use the subquery approach when you need the employee name associated with an aggregate value. It also returns multiple employees if they share the maximum salary.


In [0]:
SELECT ename, sal
FROM emp
WHERE deptno = 20
  AND sal = (SELECT MAX(sal) FROM emp WHERE deptno = 20);


ename,sal
SCOTT,3000
FORD,3000



SELECT ename, sal
FROM emp
WHERE sal = (SELECT MAX(sal) FROM emp WHERE deptno = 20); 

where this may fail ??
Good catch. It doesn’t fail, but it can return the wrong rows, because there is no deptno = 20 filter in the outer query.

The subquery gives 3000, then the outer query returns everyone in the company with sal = 3000. On your data that happens luckily to be only SCOTT and FORD (both dept 20), so it looks correct. 
But if a dept 10 employee also earned 3000, that person would appear too.

In [0]:
SELECT ename, sal
FROM emp
WHERE sal = (SELECT MAX(sal) FROM emp WHERE deptno = 20);
-- WRONG way of doing. add deptno=10 check

ename,sal
SCOTT,3000
FORD,3000


In [0]:
SELECT ename, sal
FROM emp
WHERE deptno = 20
ORDER BY sal DESC
LIMIT 1;

-- WRONG but no error.works well if no tie. here only one entry display

ename,sal
SCOTT,3000


Second highest salary along with details

using subquery
How it works:
1. The innermost query finds the highest salary.
2. The middle query finds the maximum salary below the highest salary.
3. The outer query returns all employee details for employees earning that second-highest salary.

In [0]:
SELECT *
FROM emp
WHERE sal = (
    SELECT MAX(sal)
    FROM emp
    WHERE sal < (
        SELECT MAX(sal)
        FROM emp
    )
);

empno,ename,job,mgr,hiredate,sal,comm,deptno
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20


2nd highest salary

Using DENSE rank
DENSE_RANK() assigns the same rank to employees with the same salary.

rnk = 2 returns the second-highest distinct salary, including all employees tied at that salary.

SELECT * returns all columns, including ename, sal, empno, deptno, and others.

In [0]:

SELECT *
FROM (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY sal DESC) AS rnk
    FROM emp
) t
WHERE rnk = 2;

empno,ename,job,mgr,hiredate,sal,comm,deptno,rnk
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20,2
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20,2


2nd rank dept wise

Use dense rank

In [0]:
SELECT *
FROM (
    SELECT *,
           DENSE_RANK() OVER (
               PARTITION BY deptno
               ORDER BY sal desc
           ) AS rnk
    FROM emp
) t
WHERE rnk = 2;

empno,ename,job,mgr,hiredate,sal,comm,deptno,rnk
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10,2
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20,2
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30,2


In [0]:
-- second highest salary
-- using in
use sql_catalog.sql_schema;

select max(sal) 
from emp
where sal not in (select max(sal) from emp);

max(sal)
3000


In [0]:
select max(sal) 
from emp
where sal < (select max(sal) from emp);

max(sal)
3000


In [0]:
select ename, sal
from emp
where sal < (select max(sal) from emp)
order by sal desc
limit 1;


ename,sal
SCOTT,3000


In [0]:
use sql_catalog.sql_schema;

select rownum, ename, sal
from emp
order by sal desc;
-- rownum is oracle specific

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8321045052261698>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'use sql_catalog.sql_schema;\n\nselect rownum, ename, sal\nfrom emp\norder by sal desc;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseExcepti

In [0]:
SELECT
    ROW_NUMBER() OVER (ORDER BY empno DESC) AS rownum,
    ename, empno,
    sal
FROM emp
ORDER BY sal DESC;

rownum,ename,empno,sal
6,KING,7839,5000
7,SCOTT,7788,3000
2,FORD,7902,3000
11,JONES,7566,2975
9,BLAKE,7698,2850
8,CLARK,7782,2450
13,ALLEN,7499,1600
5,TURNER,7844,1500
1,MILLER,7934,1300
12,WARD,7521,1250


In [0]:
-- how to display alternate rows or odd rows using mod i.e. % operator
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY empno) AS rn
    FROM emp
) t
WHERE rn % 2 = 1;

empno,ename,job,mgr,hiredate,sal,comm,deptno,rn
7369,SMITH,CLERK,7902,1980-12-17,800,null,20,1
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30,3
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30,5
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10,7
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10,9
7876,ADAMS,CLERK,7788,1983-01-12,1100,null,20,11
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20,13


In [0]:
-- find duplicate 

select ename from emp;

ename
SMITH
ALLEN
WARD
JONES
MARTIN
BLAKE
CLARK
SCOTT
KING
TURNER


In [0]:
select ename, count(*) from emp
group by ename;

ename,count(*)
SMITH,1
ALLEN,1
WARD,1
JONES,1
MARTIN,1
BLAKE,1
CLARK,1
SCOTT,1
KING,1
TURNER,1


In [0]:
select sal, count(*) from emp
group by sal;


sal,count(*)
800,1
1600,1
1250,2
2975,1
2850,1
2450,1
3000,2
5000,1
1500,1
1100,1


In [0]:
select sal, count(*) from emp
group by sal
having count(*) > 1;

sal,count(*)
1250,2
3000,2


In [0]:
-- display name start with M

select ename
from emp
where ename like 'M%';

ename
MARTIN
MILLER


In [0]:
-- display ename ends with 'N'
select ename
from emp
where ename like '%n'
--'n' != 'N'

ename


In [0]:
select ename
from emp
where ename like '%N'

ename
ALLEN
MARTIN


In [0]:
-- dispaly any name include m anywhere

select ename
from emp
where ename like '%M%'

ename
SMITH
MARTIN
ADAMS
JAMES
MILLER


In [0]:
-- dispaly any name not include M anywhere

select ename
from emp
where ename not like '%M%'

ename
ALLEN
WARD
JONES
BLAKE
CLARK
SCOTT
KING
TURNER
FORD
